In [1]:
#Start Code

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import json
import cv2
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
class KeypointsDataset(Dataset):
    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file, "r") as f:
            self.data = json.load(f)
        
        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def __len__ (self):
        return len(self.data)

    def __getitem__ (self, idx):
        item = self.data[idx]
        img = cv2.imread(f"{self.img_dir}/{item['id']}.png")
        h,w = img.shape[:2]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(img)
        kps = np.array(item['kps'], dtype=np.float32).flatten()

        kps[::2] * 224.0 / w
        kps[1::2] *= 224.0 / h

        return img, kps



In [3]:
val_dataset = KeypointsDataset("tennis_court_det_dataset/data/images", "tennis_court_det_dataset/data/data_val.json")
train_dataset = KeypointsDataset("tennis_court_det_dataset/data/images", "tennis_court_det_dataset/data/data_train.json")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

In [4]:
model = models.resnet50(weights=True)
model.fc = torch.nn.Linear(model.fc.in_features, 14*2)

C:\Users\LENOVO\Documents\GitHub\Build-an-AI-ML-Tennis-Analysis-system-with-YOLO-PyTorch-and-Key-Point-Extraction\.venv\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
model = model.to(device)

In [6]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)


In [ ]:
model.train()
epochs = 20

for epoch in range(epochs):
    # Notice 'imgs' here matches 'imgs.to(device)' below exactly
    for i, (imgs, kps) in enumerate(train_loader):
        imgs = imgs.to(device)
        kps = kps.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, kps)

        loss.backward()
        optimizer.step()

        if i % 10 == 0:
            print(f"Epoch {epoch}, iter {i}, loss: {loss.item()}")

Epoch 0, iter 0, loss: 256027.40625


In [ ]:
torch.save(model.state_dict(), "keypoints_model.pth")